In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
import ast

    Comme nous l'avons vu dans la partie [1_Extraction](https://github.com/ismaila-b-cisse/portfolio_dataScience/blob/master/1_extraction/main2.ipynb), nous avons un premier dataset (jeux de données) de modèles des marques de nos données e-commerces, notamment temu et darty, issu de l'extraction des modèles et leurs marques sur le site moviles. Cependant, il y a certaines marques qui ne sont pas présentes sur le site. 

    Dans ce module de la partie préparation des données, nous allons traiter ces données supplémentaires pour voir s'il y a lieu de faire des nettoyages et transformations avant de les utiliser avec nos données e-commerces.

    Avec ces nouvelles données, nous allons construire un référentiel de modèles des marques pour nos données e-commerces.


In [2]:
moviles_df = pd.read_csv('../data/extracted_data/extracted_moviles_data.csv')
# On affiche quelques lignes des données
moviles_df.head()

,marque,modele,date
0,apple,NaN,06-12-2025 18:05:19
1,apple,NaN,06-12-2025 18:05:19
2,apple,Apple iPhone 16,06-12-2025 18:05:19
3,apple,NaN,06-12-2025 18:05:19
4,apple,Apple iPhone 16 Pro,06-12-2025 18:05:19


In [ ]:
"""
    On peut ici avoir un premier apperçu des données extraites sur le site moviles. On peut voir 
    sur les colonnes les variables 'marque', 'modele' et la date d'extraction. 
    Nous avons les cinq premières observations dont toutes les marques sont 'apple', car lors de 
    l'extraction, nous avons trié les marques par ordre alphabétique.
    
    On constate à premier vue que la variable modèle a des valeurs nulles. Étant donné que le site 
    moviles classe apparemment les modèles des marques du plus récent au plus anciens, on peut 
    constater que le modèle le plus récent pour la marque 'apple' sur le site est iPhone 16. Or, 
    il y a un modèle plus récent de la marque qui est iPhone 17. Donc, nous aurons besoin de compléter
    ce dataset des données d'autres sources, peut-être, les site même des marques pour pouvoir 
    récupérer les modèles les plus récents susceptibles d'être dans nos données e-commerces.
"""

In [4]:
# Suppression de la date de l'extraction
moviles_df.drop(['date'], axis=1, inplace=True)
moviles_df.head()

,marque,modele
0,apple,NaN
1,apple,NaN
2,apple,Apple iPhone 16
3,apple,NaN
4,apple,Apple iPhone 16 Pro


# Inspection

In [5]:
moviles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5106 entries, 0 to 5105
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  5106 non-null   object
 1   modele  4621 non-null   object
dtypes: object(2)
memory usage: 79.9+ KB


In [6]:
moviles_df.describe()

,marque,modele
count,5106,4621
unique,21,4615
top,samsung,Huawei U8180 IDEOS X1
freq,1610,2


In [ ]:
"""
    Dans ce dataset, nous avons 5106 observations. 
    La variable 'marque' a 5106 valeurs, n'a pas donc pas de valeurs nulles et elle a 21 marques 
    distinctes. 
    La variable 'modele' a 4621 valeurs, donc contient des valeurs nulles comme nous l'avons déjà 
    constaté ci-dessus. Elle a 4615 modèles qui sont uniques, donc il y a des valeurs qui sont 
    dupliquées 

    Voyons d'abord les marques uniques que nous avons dans le dataset, ensuite traitons les valeurs 
    manquantes et enfin les valeurs dupliquées
"""

In [8]:
moviles_df['marque'].unique()

array(['apple', 'asus', 'beafon', 'blackview', 'crosscall', 'cubot',
       'doogee', 'doro', 'google', 'honor', 'huawei', 'motorola',
       'olympia', 'oneplus', 'oppo', 'oukitel', 'realme', 'samsung',
       'vivo', 'xgody', 'xiaomi'], dtype=object)

In [ ]:
""" 
    Nous avons 21 marques distinctes dans notre dataset. Pour rappel, voici les marques qui se 
    trouvent dans nos données initiales (notamment darty et temu) et pour lesquelles nous voulons 
    extraire les modèles dans le module moviles_scraping :
    
    ['samsung', 'apple', 'xiaomi', 'blackview', 'reborn', 'oppo', 'google', 'oukitel', 
    'honor', 'oneplus', 'artfone', 'oscal', 'fossibot', 'cubot', 'nubia', 'doogee', 
    'doro', 'vivo', 'huawei', 'hotwav', 'crosscall', 'lagoona', 'olympia', 'generique', 
    'asus', 'motorola', 'beafon', 'realme', 'iiif150', 'viqee', 'fvh', 'xgody', 
    'rainbuvvy', 'astarry']

    Cependant, comme nous avons pu le constater dans les résultats de l'extraction, ces marques 
    suivantes :
    
    ["artfone", "astarry", "fossibot", "fvh", "generique", "hotwav", 
    "iiif150", "lagoona", "nubia", "oscal", "rainbuvvy", "reborn", "viqee"] 

    ne sont présentes sur le site moviles.

    Par conséquent, nous devons trouver d'autres sources pour avoir leurs modèles comme pour les 
    modèles récents de toutes les marques ou certaines marques présentes dans notre dataset. 
"""

In [10]:
moviles_df.isnull().sum()

marque      0
modele    485
dtype: int64

In [ ]:
"""
    Nous avons 485 valeurs nulles dans les modèles. Nous allons les supprimer. 
    On peut se demander comment peut-on avoir ces valeurs nulles alors que ce sont les modèles qui 
    sont listés, donc pourquoi mettre un modèle qui n'a pas de nom ? En réalité, ces champs ne sont 
    pas liés à la présence ou non du modèle, mais à des publicités qui se trouvent entre un modèle 
    et un autre sur le site moviles. 
    C'est pourquoi il est inutile de chercher à remplir ces valeurs nulles. Donc, on les supprime.
"""

In [12]:
moviles_df = moviles_df.dropna(ignore_index=True)
moviles_df.isnull().sum() # on n'a plus de valeurs nulles
moviles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4621 entries, 0 to 4620
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  4621 non-null   object
 1   modele  4621 non-null   object
dtypes: object(2)
memory usage: 72.3+ KB


In [ ]:
"""
    Après suppression des valeurs nulles, nous avons maintenant 4621 entrées.
    Maintenant, nous allons traiter les observations dubliquées.
"""

In [14]:
moviles_df.describe()

,marque,modele
count,4621,4621
unique,21,4615
top,samsung,Huawei U8180 IDEOS X1
freq,1463,2


In [15]:
print("Nombre de lignes dupliquées : ", moviles_df.duplicated().sum())

Nombre de lignes dupliquées :  6


In [ ]:
"""
    Comme on peut bien le constater, il y a 6 lignes qui sont dupliquées. 
    Nous allons voir quelles sont ces lignes.
"""

In [17]:
moviles_df[moviles_df.duplicated(keep=False)]

,marque,modele
1165,huawei,Huawei U8180 IDEOS X1
1220,huawei,Huawei U8180 IDEOS X1
1224,huawei,Huawei U8510 IDEOS X3
1225,huawei,Huawei U8510 IDEOS X3
1386,motorola,Motorola One Zoom
1387,motorola,Motorola One Zoom
2793,samsung,Samsung Galaxy W
2794,samsung,Samsung Galaxy W
3013,samsung,Samsung Galaxy R
3021,samsung,Samsung Galaxy R


In [18]:
# Nous allons conserver les premières lignes et
# supprimer les lignes dupliquées
moviles_df[moviles_df.duplicated()]

,marque,modele
1220,huawei,Huawei U8180 IDEOS X1
1225,huawei,Huawei U8510 IDEOS X3
1387,motorola,Motorola One Zoom
2794,samsung,Samsung Galaxy W
3021,samsung,Samsung Galaxy R
4261,xgody,Xgody X25


In [ ]:
"""
    Nous allons supprimer ces duplicats
"""

In [20]:
moviles_df = moviles_df.drop_duplicates(ignore_index=True)
moviles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4615 entries, 0 to 4614
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  4615 non-null   object
 1   modele  4615 non-null   object
dtypes: object(2)
memory usage: 72.2+ KB


In [21]:
moviles_df.describe()

,marque,modele
count,4615,4615
unique,21,4615
top,samsung,Apple iPhone 16
freq,1461,1


In [ ]:
"""
    À présent, nous pouvons voir qu'il n'y a plus de valeurs nulles dans la variables modèles, non
    plus d'observation dupliquée dans le dataset.
"""

In [23]:
moviles_df.head()

,marque,modele
0,apple,Apple iPhone 16
1,apple,Apple iPhone 16 Pro
2,apple,Apple iPhone 16 Pro Max
3,apple,Apple iPhone 16 Plus
4,apple,Apple iPhone 15


In [24]:
moviles_df.index

RangeIndex(start=0, stop=4615, step=1)

In [ ]:
"""
    Nous allons maintenant supprimer le nom de la marque dans les valeurs de la variable 'modele'
"""

In [ ]:
for i in range(len(moviles_df)):
    marque = moviles_df['modele'].str.split().str.get(0)[i]
    moviles_df['modele'][i] = moviles_df['modele'][i].lower().replace(marque.lower(), '').strip()

In [27]:
moviles_df.head(5)

,marque,modele
0,apple,iphone 16
1,apple,iphone 16 pro
2,apple,iphone 16 pro max
3,apple,iphone 16 plus
4,apple,iphone 15


In [ ]:
""" 
    Maintenant, nous allons compléter ce dataset par d'autres données sur les marques qui n'y sont pas
    présentes, mais aussi les modèles récents des marques qui y sont présentes, si nécessaire.

    Commençons d'abord par compléter les modèles des marques qu'on a déjà. Pour cela, affichons les 
    premières occurrences de chaque marque et son modèle. Pour rappel, les marques sont par ordre
    alphabétique dans le dataset. Les modèles de chaque marque sont classés du plus récent au plus
    ancien, comme ils le sont apparemment sur le site moviles, donc la première occurrence d'une 
    marque m a son modèle le plus récent dès sa première apparition dans le dataset à l'index i.
"""

In [29]:
first_o = moviles_df.duplicated(subset=['marque'], keep="first")
type(first_o)
first_o_indexes = []
for i, b in enumerate(first_o):
    if b==False:
        first_o_indexes.append(i)
    else:
        pass

moviles_df.iloc[first_o_indexes]

,marque,modele
0,apple,iphone 16
58,asus,zenfone 9
199,beafon,m6
258,blackview,a50 2022
362,crosscall,action-x5
387,cubot,p50
498,doogee,s98
647,doro,primo 368
738,google,pixel 9 pro fold
766,honor,x8c


In [ ]:
"""
    On peut voir chaque marque et son modèle le plus récent dans notre dataset. Ainsi, nous pouvons 
    savoir pour quelles marques doit-on compléter ses modèles les plus récents pour compléter notre 
    référentiel. 
"""



In [31]:
moviles_df.loc[moviles_df['marque']=='astarry'].head(60)
#moviles_df.loc[moviles_df['marque']=='vivo'].tail(30)

,marque,modele


In [ ]:
"""
    Ci-dessus, on sélectionne chaque marque et certains de ces modèles pour voir s'il est nécessaire 
    de procéder à un ajout d'autres modèles ou non.
"""

In [33]:
"""
    Mais pour s'assurer qu'on a laissé les modèles récents d'aucune marque de côté, on va vérifier, 
    lors de cette nouvelle collecte, toutes les marques qu'on a, pour l'instant, dans cette liste :
    
    ['samsung', 'apple', 'xiaomi', 'blackview', 'reborn', 'oppo', 'google', 'oukitel', 
    'honor', 'oneplus', 'artfone', 'oscal', 'fossibot', 'cubot', 'nubia', 'doogee', 
    'doro', 'vivo', 'huawei', 'hotwav', 'crosscall', 'lagoona', 'olympia', 'generique', 
    'asus', 'motorola', 'beafon', 'realme', 'iiif150', 'viqee', 'fvh', 'xgody', 
    'rainbuvvy', 'astarry']  

    Cette collecte est fait manuellement.
"""
load_dotenv()
ADD_MODEL_DIC = os.getenv("ADD_MODEL_DIC")
add_model_dic = ast.literal_eval(ADD_MODEL_DIC)

In [ ]:
"""
    En dehors d'ajouter les modèles les plus récents à ceux des marques du dataset moviles, nous
    avons exploré d'autres sources également, comme évoqué ci-dessus, pour avoir les modèles des 
    marques qui n'y sont pas présentes. Ces sources sont en premier lieu les sites des marques, 
    ensuite d'autres sources à chaque fois qu'il est nécessaire. Certaines marques, même si elles 
    restent une exception, n'ont pas de site officiel. Dans ce cas, entre autres, je peux utiliser 
    les sites e-commerces, entre autres sources, qui vendent leurs produits. 
    
    Une autre manière simple et rapide d'avoir les modèles des marques est de les faire générer par 
    une IA generative comme chatGPT, mais dans le cadre de ce portfolio, nous avons choisi la première
    solution qui vient d'être expliquée. En plus, même si elles seraient générées, elles ne seront pas
    dispensées de vérification pour voir s'il y a pas des incohérences, des modèles inventés pour des 
    marques qui en réalité n'exite pas, etc.
    
    Après la collecte, j'ai instancié un dictionnaire et l'ai initialisé avec ces nouvelles données,
    ensuite, nous l'affectons à un dataframe pour leur éventuel nettoyage avant de les concatener au 
    dataset moviles pour avoir un référentiel de modèles et leurs marques dont on a besoin dans la
    préparation de nos données e-commerces.
"""

In [35]:
brand_list = []
model_list = []
for brand, m_list in add_model_dic.items():
    for i, model in enumerate(m_list):
        brand_list.append(brand)
        model_list.append(model)


brand_s = pd.Series(brand_list)
model_s = pd.Series(model_list)
    
add_model_df = pd.DataFrame(columns=['marque', 'modele'])
add_model_df['marque']=brand_s
add_model_df['modele']=model_s
add_model_df = add_model_df.sort_values(by=['marque'], ignore_index=True)
add_model_df

,marque,modele
0,apple,iPhone 17 Pro
1,apple,iPhone 17
2,apple,iPhone Air
3,apple,iPhone 17 Pro Max
4,artfone,G6
...,...,...
491,xiaomi,REDMI 15C
492,xiaomi,Xiaomi 15 Ultra
493,xiaomi,Xiaomi 15T
494,xiaomi,POCO F8 Pro


In [ ]:
"""
    Comme nous pouvons le constater, nous avons maintenant le modèle iphone 17, par exemple, pour
    la marque Apple, alors qu'il n'est pas présent dans le dataset de moviles.

    Maintenant, inspectons ces nouvelles données
"""

In [37]:
# Inspecons nos nouvelles données
add_model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 496 entries, 0 to 495
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  496 non-null    object
 1   modele  496 non-null    object
dtypes: object(2)
memory usage: 7.9+ KB


In [38]:
add_model_df.describe()

,marque,modele
count,496,496
unique,30,480
top,rainbuvvy,ROG Phone 9
freq,58,3


In [ ]:
""" 
    Nous pouvons voir que nous avons 496 lignes, qui n'ont pas de valeurs nulles, avec 30 marques
    uniques et 480 modèles uniques.
    Supprimons les doublons
"""

In [40]:
add_model_df[add_model_df.duplicated()]

,marque,modele
20,asus,ROG Phone 9
21,asus,ROG Phone 9
25,asus,ROG Phone 8
28,asus,ROG Phone 7
143,fossibot,f105
191,hotwav,t7 pro
350,rainbuvvy,xs15 pro
356,rainbuvvy,xs15
360,rainbuvvy,xs16
364,rainbuvvy,s9 pro


In [41]:
add_model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 496 entries, 0 to 495
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  496 non-null    object
 1   modele  496 non-null    object
dtypes: object(2)
memory usage: 7.9+ KB


In [42]:
add_model_df = add_model_df.drop_duplicates(ignore_index=True)
add_model_df.describe()

,marque,modele
count,481,481
unique,30,480
top,rainbuvvy,c80
freq,52,2


In [ ]:
"""
    On peut voir qu'on a maintenant que la taille du nouveau est de 481 lignes, donc les doublons
    ont été supprimés. 
    Il y a le modèle c80 qui est dupliqué. Autrement dit, il y a deux marques différentes qui ont 
    chacune un modèle qui porte le nom c80.
    Quelles sont ces marques ?
"""

In [44]:
add_model_df[add_model_df['modele'].duplicated(keep=False)]

,marque,modele
26,beafon,c80
267,oscal,c80


In [ ]:
"""
    Ce sont les marques 'beafon' et 'oscal' qui ont chacune un modèle c80.
"""

In [ ]:
""" 
    Dès lors qu'il n'y a plus d'observation dupliquée, faison une concatenation avec le dataset 
    de moviles. Ensuite, faisons un nettoyage du dataset final, notamment la suppressions des 
    éventuels doublons.

"""

In [48]:
reference_df = pd.concat([moviles_df, add_model_df], ignore_index=True)
reference_df

,marque,modele
0,apple,iphone 16
1,apple,iphone 16 pro
2,apple,iphone 16 pro max
3,apple,iphone 16 plus
4,apple,iphone 15
...,...,...
5091,xiaomi,REDMI 15
5092,xiaomi,Xiaomi 15 Ultra
5093,xiaomi,Xiaomi 15T
5094,xiaomi,POCO F8 Pro


In [49]:
reference_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5096 entries, 0 to 5095
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  5096 non-null   object
 1   modele  5096 non-null   object
dtypes: object(2)
memory usage: 79.8+ KB


In [50]:
reference_df.describe()

,marque,modele
count,5096,5096
unique,31,4878
top,samsung,x6
freq,1471,5


In [ ]:
"""
    On peut voit nous avons 5096 entrées. Le dataset est complet, il n'y a pas de valeurs nulles.
    Nénamoins, nous avons des valeurs dupliqués. Nous pouvons également voir que les modèles n'ont
    pas le même format d'écriture. En effet, certains sont en minuscule, d'autres en majuscule et 
    d'auttres encore en majuscule et minuscule.
    
    Avant de traiter ces erreurs, voyons d'abord comment nos données sont concatener. Où se trouve 
    par exemple les modèles récents de la marque 'apple' qu'on vient de concatener au dataset moviles 
    afin de s'asurer qu'ils sont bien ajoutés dans le dataset final, même si dans l'affichage 
    ci-dessus, on peut déjà voir que les derniers modèles de la marque 'xiaomi', qui ne sont pas 
    présent dans le dataset moviles, sont bien présents dans le dataset concatené ?
"""

In [52]:
reference_df.loc[reference_df['marque']=="apple"]

,marque,modele
0,apple,iphone 16
1,apple,iphone 16 pro
2,apple,iphone 16 pro max
3,apple,iphone 16 plus
4,apple,iphone 15
...,...,...
57,apple,iphone 16gb
4615,apple,iPhone 17 Pro
4616,apple,iPhone 17
4617,apple,iPhone Air


In [ ]:
""" 
    Nous pouvons voir qu'ils sont bien dans le dataset, mais comme c'est une concaténation, le 
    fonction a placé le deuxième dataset à la fin du premier. Nous pouvons changé cela en triant 
    le dataset.
    Avant de faire cela, harmonisons les marques et les modèles en leur rendant tous minuscule et 
    supprimant les éventuels de début et de fin
"""

In [62]:
reference_df['marque'] = reference_df['marque'].str.lower().str.strip()
reference_df['modele'] = reference_df['modele'].str.lower().str.strip()
#sorted(reference_df['modele'].unique()[:60])
reference_df

,marque,modele
0,apple,iphone 11
1,apple,iphone 11 pro
2,apple,iphone 11 pro max
3,apple,iphone 12
4,apple,iphone 12 mini
...,...,...
5091,xiaomi,redmi s2
5092,xiaomi,redmi y3
5093,xiaomi,xiaomi 15 ultra
5094,xiaomi,xiaomi 15t


In [55]:
reference_df = reference_df.sort_values(by=['marque','modele'], ignore_index=True)
reference_df.head()

,marque,modele
0,apple,iphone 11
1,apple,iphone 11 pro
2,apple,iphone 11 pro max
3,apple,iphone 12
4,apple,iphone 12 mini


In [ ]:
"""
    Après la concatenation des deux datasets, nous avons trié les observations, comme nous pouvons
    le constater dans l'affichage des données. Pour voir plus clair, affichons que les modèles de la
    première marque, par ordre alphabétique, dans notre dataset qui est ici 'apple'. Ci-dessus, on a
    déjà vu que les modèles des nouvelles données sont ajoutés à partir de l'indice 4615, donc la 
    fin de la dataset de moviles. 
    Maintenant, revoyons où se sitient-ils après le tri
"""

In [66]:
reference_df.loc[reference_df['marque']=="apple"][20:30]

,marque,modele
20,apple,iphone 16 plus
21,apple,iphone 16 pro
22,apple,iphone 16 pro max
23,apple,iphone 16gb
24,apple,iphone 17
25,apple,iphone 17 pro
26,apple,iphone 17 pro max
27,apple,iphone 3g 16gb
28,apple,iphone 3g 8gb
29,apple,iphone 3gs 16gb


In [ ]:
""" 
    On peut voir que les modèles 'phone 17', 'iphone 17 pro', etc ont changé d'indices pour être 
    parmi les premières indices. Ainsi, tous les modèles  de la marque 'apple' sont maintenant au 
    même endroit dans les dataset.

    Après avoir uniformisé le format d'écriture, supprimé les éventuels espace de débuts et de fin,
    traitons à présent les observations dupliquées.
    Commé évoqué plus haut, après l'inspection des données, on peut voir que le dataset final contient
    des doublons. Déterminons lesquels et traitons-les en conséquence
"""

In [67]:
# nombre de doublons
len(reference_df[reference_df.duplicated()])

19

In [70]:
reference_df[reference_df.duplicated()]

,marque,modele
272,beafon,sl495
330,blackview,bl6000 pro
346,blackview,bv4900
367,blackview,bv6300 pro
380,blackview,bv8800
507,cubot,j10
531,cubot,kingkong mini 2
610,cubot,x30
2215,oppo,a40
2217,oppo,a5


In [71]:
# suppression des duplicats
reference_df.drop_duplicates(inplace=True, ignore_index=True)

In [72]:
reference_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5077 entries, 0 to 5076
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  5077 non-null   object
 1   modele  5077 non-null   object
dtypes: object(2)
memory usage: 79.5+ KB


In [74]:
reference_df.describe()

,marque,modele
count,5077,5077
unique,31,4854
top,samsung,x20
freq,1470,5


In [ ]:
"""
    La taille de notre référentiel de modèles de marques est désormais passée de 5096 à 5077 après
    la suppression des lignes dupliquées.
    Cependant, des valeurs de modèle peuvent être dupliqués comme c'est le cas du modèle 'x20' qui 
    est présent 5 fois dans la variable modèle. Cela est tout à fait compréhensible dans la mesure
    où deux marques peuvent avoir un même nom pour leurs modèles respectifs, comme expliqué ci-dessus.
    C'est pour pourquoi prendre comme clé (marque, modele), lors du tri, et les avoir tous dans le 
    dataset est plus pertinent que de prendre seulement l'une des deux variables. 
    Car si on prenait seulement les modèles sans les marques, par exemple, quand on cherchera dans nos 
    données e-commerces, qu'un modèle donné est présent ou non, on peut bien trouver que ce modèle 
    est présent pour un produit, mais comment peut-on s'assurer que ce modèle n'est pas écrit par 
    erreur par l'utilissateur et qu'il appartient réellement à cette marque ? 
    C'est là où avoir un référenctiel des modèles et leurs marques à toute son importance. 
"""

In [ ]:
""" 
    Enfin, nous pouvons exporter le référentiel dans un fichier csv
"""

In [80]:
reference_df.to_csv("../data/cleaned_data/reference_data.csv", index=False)
print("===== Données chargées")

===== Données chargées
